<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/sample-answers/data_analysis_assignment_answers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Foundations of Statistical Inference & Hypothesis Testing**

Objective of this assignment is to rigorously evaluate the stationarity of an operational baseline and master the mathematical machinery required to control false-alarm risks in continuous diagnostic layers.

## **Part A: Theoretical Fundamentals (Maximum Likelihood & Decision Space)**



In an operational micro-scale anomaly detection boundary is constructed by approximating the unobservable multivariate population parameters $(\boldsymbol{\mu}, \boldsymbol{\Sigma})$ with their corresponding unbiased empirical sample estimators $(\widehat{\boldsymbol{\mu}}_n, \mathbf{S})$.
* Formally derive why the raw Maximum Likelihood Estimator for the covariance matrix, $\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}$, is inherently **biased** in finite samples, showing that:

$$\mathbb{E}[\widehat{\boldsymbol{\Sigma}}_{\text{MLE}}] = \frac{n-1}{n}\boldsymbol{\Sigma}$$


* Mathematically demonstrate how applying Bessel’s correction yields the unique, unbiased estimator $\mathbf{S}$.


2. Under the null hypothesis $H_0$ that a structural asset remains completely healthy (the status quo), a chosen test statistic tracks a deterministic parametric distribution.
* Define a **Type I Error ($\alpha$)** and a **Type II Error ($\beta$)** explicitly within the context of continuous structural health monitoring.
* If an engineer sets an ultra-conservative significance threshold (e.g., $\alpha = 0.0001$) to prevent false alarms from shutting down an industrial plant, explain the mathematical consequence this decision exerts on the statistical power of the diagnostic layer. How does this modify the geometric volume of your "healthy operation" confidence ellipsoid?

## **Part B: Theoretical Extension (Asymptotic Distributions & Slutsky's Theorem)**

The Lindeberg-Lévy Multivariate Central Limit Theorem establishes that as the sample snapshot size approaches infinity ($n \to \infty$), the standardized difference between the empirical mean and population mean converges in distribution to a standard Multivariate Gaussian:

$$\sqrt{n}(\widehat{\boldsymbol{\mu}}_n - \boldsymbol{\mu}) \xrightarrow{d} \mathscr{N}(\mathbf{0}, \boldsymbol{\Sigma})$$



However, the true population covariance matrix $\boldsymbol{\Sigma}$ is a latent, unobservable parameter.
* State **Slutsky’s Theorem** rigorously.
* Prove how Slutsky's Theorem justifies substituting the empirical sample covariance matrix $\mathbf{S}$ for the latent population matrix $\boldsymbol{\Sigma}$ as $n \to \infty$, thereby operationalizing the practical parametric modeling distribution used in your dashboard:

$$\widehat{\boldsymbol{\mu}}_n \sim \mathscr{N}\left(\widehat{\boldsymbol{\mu}}_n, \frac{1}{n}\mathbf{S}\right)$$







## **Part C: Numerical Verification**

***Scenario:** You are provided an industrial dataset containing $n = 5,000$ snapshots across $m = 3$ sensor channels measuring physical strain. To verify the weak i.i.d. assumption before initializing a global PCA or Factor Analysis layer, you must explicitly test **Condition 1: Constant Joint Mean (First Moment Homogeneity)** across the time index.*

**Your Task:**
Write a clean, self-contained Python function within Google Colab to partition the dataset into $g = 5$ consecutive, non-overlapping blocks (chunks) of size $n_j = 1,000$. Implement a Multivariate Analysis of Variance (MANOVA) tracking framework from scratch (or leveraging `statsmodels`) to calculate **Wilks' Lambda ($\Lambda$)**.

$$\Lambda = \frac{|\mathbf{W}|}{|\mathbf{B} + \mathbf{W}|}$$

Transform your calculated $\Lambda$ using **Bartlett’s $\chi^2$ approximation**:


$$\chi^2_{\text{calc}} = -\left(n - 1 - \frac{m + g}{2}\right) \ln \Lambda$$

Execute your code using the synthetic data generator provided below. Report your exact calculated Wilks' $\Lambda$, the Bartlett $\chi^2$ test statistic, and its corresponding $p$-value under $\chi^2 \sim \chi^2_{m(g-1)}$. Based on an alpha threshold of $\alpha = 0.05$, state your final diagnostic conclusion: Has the baseline center shifted over time, or is the first moment homogeneous?



```python
import numpy as np
import pandas as pd
import scipy.stats as stats

# --- DO NOT MODIFY: Synthetic Strain Sensor Data Generator ---
np.random.seed(42)
n_samples = 5000
n_features = 3

# Generating base stationary multivariate normal noise
base_data = np.random.multivariate_normal(
    mean=[0.5, -0.2, 1.1],
    cov=[[0.09, 0.02, 0.01],
         [0.02, 0.06, 0.03],
         [0.01, 0.03, 0.05]],
    size=n_samples
)

# Injecting a subtle, hidden structural drift in the final 1,000 snapshots
base_data[4000:, 0] += 0.015  # Sensor 1 drift
base_data[4000:, 2] -= 0.010  # Sensor 3 drift

df_strain = pd.DataFrame(base_data, columns=['Strain_Ch1', 'Strain_Ch2', 'Strain_Ch3'])
# ----------------------------------------------------------------

def verify_first_moment_homogeneity(df: pd.DataFrame, g_chunks: int = 5) -> dict:
    """
    Partitions the dataset into g chunks and evaluates first-moment homogeneity
    via Wilks' Lambda and Bartlett's Chi-Square asymptotic transformation.
    """
    # YOUR CODE GOES HERE
    # 1. Split df into g equal consecutive blocks
    # 2. Compute the global mean and individual chunk means
    # 3. Calculate Within-Chunk Matrix (W) and Between-Chunk Matrix (B)
    # 4. Compute Wilks' Lambda, Bartlett's Chi-Square, and the final p-value
    pass

# Execute your completed function below:
# results = verify_first_moment_homogeneity(df_strain, g_chunks=5)
# print(results)
```

# **Geometric Subspace Optimization via Principal Component Analysis (PCA)**

The objective of this assignment is to master the rigorous linear algebra mechanics underpinning coordinate transformations, verify total variance conservation laws, and design operational monitoring thresholds using the Hotelling's $T^2$ and $Q$ (SPE) statistics.

## **Part A: Theoretical Fundamentals (Coordinate Projections & Orthogonality)**

1. Let $\widetilde{\mathbf{X}}_i = (\mathbf{X}_i - \boldsymbol{\mu}) \in \mathbb{R}^m$ represent a zero-mean, centered baseline feature vector with a symmetric, positive-semidefinite population covariance matrix $\boldsymbol{\Sigma} = \mathbb{E}[\widetilde{\mathbf{X}}_i \widetilde{\mathbf{X}}_i^T]$. We apply the spectral decomposition $\boldsymbol{\Sigma} = \mathbf{P} \mathbf{\Lambda} \mathbf{P}^T$, and define the orthogonal projection mapping:

$$\mathbf{Z}_i = \mathbf{P}^T \widetilde{\mathbf{X}}_i$$


* Prove step-by-step that the theoretical covariance matrix of the transformed vector, $\mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T]$, simplifies exactly to the strictly diagonal matrix $\mathbf{\Lambda}$.
* Explain the precise geometric and statistical meaning of this diagonal result regarding the cross-covariances between distinct coordinates $Z_{i,j}$ and $Z_{i,k}$ (for $j \neq k$).


2. The total variance of a multivariate system can be measured in its raw coordinate space or its transformed principal component space.
* Utilizing the algebraic properties of the matrix trace ($\text{tr}$), specifically its invariance under cyclic permutations, formally prove that the total system variance is preserved during the mapping from raw space to the de-correlated principal space:

$$\text{tr}(\boldsymbol{\Sigma}) = \text{tr}(\mathbf{\Lambda}) = \sum_{j=1}^m \lambda_j$$


* If we partition the empirical matrix into an explained subspace ($\widehat{\mathbf{P}}_k \in \mathbb{R}^{m \times k}$) and an unexplained residual subspace ($\widehat{\mathbf{P}}_{m-k} \in \mathbb{R}^{m \times (m-k)}$), state the rigorous mathematical formulations for the **Cumulative Explained Variance Ratio $\Phi(k)$** and the **Residual Unexplained Variance Ratio $\Psi(k)$**.


3. In structural health monitoring (SHM), we truncate our coordinate system to retain only the first $k$ components ($k < m$). The centered raw vector is partitioned as $(\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n) = \widehat{\mathbf{P}}_k \mathbf{z}_{i,k} + \widehat{\mathbf{P}}_{m-k} \mathbf{z}_{i,m-k}$.
* Define the reconstructed vector $\widehat{\mathbf{x}}_i$ and the residual error vector $\mathbf{e}_i$.
* Prove via the Pythagorean identity that the total squared Euclidean norm of the centered observation vector cleanly decomposes into independent subspace energies:

$$\|\mathbf{x}_i - \widehat{\boldsymbol{\mu}}_n\|^2 = \|\mathbf{z}_{i,k}\|^2 + \|\mathbf{z}_{i,m-k}\|^2$$


* Contrast the diagnostic functions of Hotelling's $T^2$ statistic and the residual $Q$ statistic (SPE). If a structure encounters an environmental load anomaly (e.g., extreme operational wind loads) versus an internal structural fracture (e.g., a fatigue crack changing the structural load pathways), match each event type to the statistic ($T^2$ or $Q$) most sensitive to it and justify your choices.

## **Part B: Numerical Verification**



***Scenario:** You are building a real-time diagnostics layer for a structural component monitored by $m = 4$ sensors. You have a baseline archive of $n = 3,000$ nominal snapshots. To establish operational boundaries, you must write an optimization pipeline that analyzes how Hotelling’s $T^2$ and the residual $Q$ statistic respond to different truncation boundaries $k \in \{1, 2, 3\}$.*

**Your Task:**
Complete the Python class method below inside Google Colab. The function must:

1. Standardize and center the incoming realizations.
2. Decompose the unbiased sample covariance matrix $\mathbf{S}$ using `np.linalg.eigh` and sort components in descending order ($\hat{\lambda}_1 \ge \dots \ge \hat{\lambda}_m$).
3. Loop through every possible subspace cutoff value $k \in \{1, 2, 3\}$, calculating the sample vector array for Hotelling’s $T^2$ ($T^2_i = \sum_{j=1}^k \frac{z_{i,j}^2}{\hat{\lambda}_j}$) and the $Q$ statistic ($Q_i = \sum_{j=k+1}^m z_{i,j}^2$).
4. Compute the expected value (empirical mean) of $T^2$ and $Q$ across all samples for each $k$.
5. Return these metrics to verify your theoretical derivations in Part A.

Execute your code using the baseline generator provided below. Report your calculated mean $T^2$ profiles and mean $Q$ profiles. Explain why the mean $T^2$ value follows a predictable integer climb ($\mathbb{E}[T^2] = k$) under nominal conditions.

```python
import numpy as np
import pandas as pd
from typing import Optional, Sequence, Dict, Any

# --- DO NOT MODIFY: Baseline Structural Sensor Data Generator ---
np.random.seed(101)
n_samples = 3000
m_features = 4

# Simulating nominal structural vibrations across 4 channels
vibration_data = np.random.multivariate_normal(
    mean=[0.0, 0.0, 0.0, 0.0],
    cov=[[1.20, 0.45, 0.30, 0.15],
         [0.45, 0.90, 0.25, 0.10],
         [0.30, 0.25, 0.75, 0.40],
         [0.15, 0.10, 0.40, 0.60]],
    size=n_samples
)
df_vibe = pd.DataFrame(vibration_data, columns=['S1_Acc', 'S2_Acc', 'S3_Acc', 'S4_Acc'])
# ----------------------------------------------------------------

def optimize_pca_subspaces(df: pd.DataFrame, columns: Optional[Sequence[str]] = None) -> Dict[str, Any]:
    """
    Performs PCA on the data frame and computes the profile of Hotelling's T^2
    and residual Q (SPE) statistics for all possible truncation horizons k.
    """
    if columns is None:
        target_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    else:
        target_cols = list(columns)
        
    X = df[target_cols].copy().dropna().values
    n, m = X.shape
    
    # 1. Micro-scale Centering
    mu_hat = np.mean(X, axis=0)
    X_centered = X - mu_hat
    
    # 2. Compute Unbiased Sample Covariance Matrix S (ddof=1)
    S_matrix = np.cov(X_centered, rowvar=False, ddof=1)
    
    # 3. Spectral Decomposition & Descending Sort
    # YOUR CODE GOES HERE
    # Hint: Use np.linalg.eigh, sort eigenvalues/eigenvectors, and clip at 1e-15
    lambda_hat = np.zeros(m)  # Replace with actual computation
    P_hat = np.zeros((m, m))  # Replace with actual computation
    
    # 4. Transform observations to Score Space Z
    Z_scores = np.zeros((n, m))  # Replace with actual computation
    
    # 5. Evaluate Statistics profiles across k = 1 to m-1
    k_values = np.arange(1, m)
    mean_T2_profile = []
    mean_Q_profile = []
    
    for k in k_values:
        # Calculate Hotelling's T^2 samples for current k selection
        # T2 = sum_{j=1}^k (z_j^2 / lambda_j)
        # Calculate Q statistic samples for current k selection
        # Q  = sum_{j=k+1}^m (z_j^2)
        
        # YOUR CODE GOES HERE
        
        mean_T2_profile.append(0.0) # Replace with empirical mean
        mean_Q_profile.append(0.0)  # Replace with empirical mean

    return {
        "eigenvalues": lambda_hat,
        "k_values": k_values,
        "mean_T2": np.array(mean_T2_profile),
        "mean_Q": np.array(mean_Q_profile)
    }

# Run execution layer
# results = optimize_pca_subspaces(df_vibe)
# print("Eigenvalues:", results["eigenvalues"])
# print("k truncation options:", results["k_values"])
# print("Mean T^2 Profile:", results["mean_T2"])
# print("Mean Q Profile:", results["mean_Q"])

```

# **Latent Subspace Decomposition via Factor Analysis (FA)**

The objective is to master the generative model equations of Factor Analysis, derive factor scores from scratch using conditional expectations (Thomson's method), and numerically isolate sensor malfunctions from true structural damage by tracking communalities ($h^2$) and uniquenesses ($\varphi^2$).

## **Part A: Theoretical Fundamentals (Generative Model & Commonalities)**

1. Consider your structured data setup where $\mathbf{Z}_i \in \mathbb{R}^m$ is a normalized, zero-mean baseline feature vector whose population covariance matrix represents the universal correlation matrix $\mathbf{R} = \mathbb{E}[\mathbf{Z}_i \mathbf{Z}_i^T]$. The Factor Analysis model posits the linear generative structure:
$$\mathbf{Z}_i = \boldsymbol{\Lambda} \mathbf{F}_i + \boldsymbol{\epsilon}_i$$
where $\mathbf{F}_i \in \mathbb{R}^k$ are the orthogonal common factors ($\mathbb{E}[\mathbf{F}_i\mathbf{F}_i^T] = \mathbf{I}_{k \times k}$) and $\boldsymbol{\epsilon}_i \in \mathbb{R}^m$ is the diagonalized measurement noise matrix ($\mathbb{E}[\boldsymbol{\epsilon}_i\boldsymbol{\epsilon}_i^T] = \boldsymbol{\Psi} = \text{diag}(\varphi_1^2, \dots, \varphi_m^2)$).
* Utilizing the constraint that the latent factors and errors are independent ($\mathbb{E}[\boldsymbol{\epsilon}_i \mathbf{F}_i^T] = \mathbf{0}$), prove the **Fundamental Equation of Factor Analysis**:

$$\mathbf{R} = \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$$


* Focus on the $j$-th diagonal entry of this decomposition. Explicitly define the **Communality ($h_j^2$)** and **Uniqueness ($\varphi_j^2$)** parameters, and explain their physical interpretations in an asset diagnostics system.


2. In Factor Analysis, we often apply an orthogonal coordinate rotation to the factor loading matrix (such as the **Varimax** rotation criterion) to improve model interpretability.
* Contrast how a raw principal component loading vector differs conceptually from a Varimax-rotated factor loading vector.
* Explain why orthogonal rotations can alter the individual columns of the loading matrix $\boldsymbol{\Lambda}$ without changing the calculated communality ($h_j^2$) of any sensor or modifying the global covariance approximation $\mathbf{R} \approx \boldsymbol{\Lambda}\boldsymbol{\Lambda}^T + \boldsymbol{\Psi}$.

## **Part B: Theoretical Extension (Factor Estimation through Conditional Expectation)**



1. Because the common factors $\mathbf{F}_i$ are unobservable latent parameters, they cannot be directly extracted via geometric projections like PCA. Instead, we must treat them as hidden random vectors and calculate their realization using **Thomson’s Regression Method**, which relies on the conditional expectation $\mathbf{f}_i = \mathbb{E}[\mathbf{F}_i | \mathbf{z}_i]$.
* Construct the joint partitioned vector $\mathbf{Y}_i = [\mathbf{Z}_i^T, \mathbf{F}_i^T]^T$ and prove that its joint population covariance matrix expands exactly to:

$$\boldsymbol{\Sigma}_{YY} = \begin{bmatrix} \mathbf{R} & \boldsymbol{\Lambda} \\ \boldsymbol{\Lambda}^T & \mathbf{I}_{k \times k} \end{bmatrix}$$


* Under the standard assumption of a Multivariate Normal distribution, the conditional mean of a partitioned system is governed by the projection identity $\mathbb{E}[\mathbf{X}_2 | \mathbf{x}_1] = \boldsymbol{\mu}_2 + \boldsymbol{\Sigma}_{21} \boldsymbol{\Sigma}_{11}^{-1} (\mathbf{x}_1 - \boldsymbol{\mu}_1)$. Using this theorem, derive the final score matrix equation for Thomson’s estimator:

$$\mathbf{f}_i = \boldsymbol{\Lambda}^T \mathbf{R}^{-1} \mathbf{z}_i$$


# **Subspace Diagnostics & Feature De-correlation**

In structural health monitoring (SHM) and industrial asset diagnostics, multi-sensor grids generate highly continuous, collinear data streams. To establish reliable anomaly detection limits, engineers must project these high-dimensional observation spaces onto lower-dimensional sub-spaces.

This assignment evaluates two fundamental methods for dimensionality reduction using an identical 4-channel sensor array (`Sensor_1` to `Sensor_4`):

1. **Principal Component Analysis (PCA):** A geometric, non-parametric approach aimed at total variance maximization.
2. **Factor Analysis (FA):** A generative, parametric framework aimed at partitioning shared structural variance from unique localized sensor noise.



## **Develop a Feature Analysis Dashboard**

**UI Layout 1: PCA Optimization Suite ($2\times  3$ Horizontal Layout)**

The first interface must display a 5-panel horizontal grid layout.
* **Panel 1,1 (Heatmap):** Feature Loadings Matrix $|P_{j,r}|$.
* **Panel 1,2 (Bar Chart):** Absolute Eigenvalues ($\lambda_j$) tracking variance magnitude.
* **Panel 1,3 (Combined Bar + Line):** Marginal Explained Variance % per axis (bars) overlaid with Cumulative Captured Variance Curve (dashed line).
* **Panel 2,1 (Bar Chart):** Residual Space (Unexplained Variance %) showing excluded information ($100 - \text{Cumulative Variance \%}$).
* **Panel 2,2 (Scatter Line):** Sample Mean Hotelling's $T^2$ tracking distance vs. Subspace Size $k$.
* **Panel 2,3 (Scatter Line):** Sample Mean $Q$ Statistic (SPE) tracking residual decay vs. Truncation Cutoff $k$.

**UI Layout 2: FA Latent Subspace Suite ($2 \times 2$ Grid Layout)**

The second interface must structure its subplots into a balanced $2 \times 2$ matrix.

* **Panel 1,1 (Heatmap):** Structural Loadings Matrix $|\lambda_{j,r}|$.
* **Panel 1,2 (Stacked Horizontal Bar):** Variance Allocation breakdown showing Shared Communality ($h^2$) versus Channel Uniqueness ($\varphi^2$) totaling $100\%$ per sensor.
* **Panel 2,1 (Scatter Line):** Sensor Uniqueness Noise Floor Profile ($\varphi^2$) across monitored channels to instantly isolate instrument faults.
* **Panel 2,2 (Bar Chart):** Latent Factor Scores Empirical Variance showing the post-rotation power of the extracted physical dimensions.


## Analysis

### Generate the following synthetic data

The synthetic multi-sensor asset data must be generated using
```python
import numpy as np
import pandas as pd

# Set execution seed for perfect reproducible properties
np.random.seed(42)
n_samples = 2500

# 1. Generate underlying, independent true physical driving forces (Latent Factors)
f1 = np.random.normal(0, 1, n_samples)  # Primary structural mode/load
f2 = np.random.normal(0, 1, n_samples)  # Secondary operational mode

# 2. Construct the Observable Multi-Sensor Array with targeted noise profiles
# Sensors 1 and 2 couple strongly to the primary process (f1)
s1 = 0.85 * f1 + 0.10 * f2 + np.random.normal(0, 0.3, n_samples)
s2 = 0.80 * f1 + 0.15 * f2 + np.random.normal(0, 0.35, n_samples)

# Sensor 3 couples strongly to the secondary process (f2)
s3 = 0.12 * f1 + 0.90 * f2 + np.random.normal(0, 0.25, n_samples)

# Sensor 4 suffers an extreme electrical/instrumentation fault (massive localized white noise)
s4 = 0.02 * f1 + 0.05 * f2 + np.random.normal(0, 1.40, n_samples)

# 3. Compile into the final evaluation DataFrame
df_asset = pd.DataFrame(
    data=np.vstack([s1, s2, s3, s4]).T,
    columns=['Sensor_1', 'Sensor_2', 'Sensor_3', 'Sensor_4']
)

```

**Empirical Covariance and Correlation Properties**

* **Sensors 1, 2, and 3:** High cross-correlation due to strong coupling with the common underlying physical structures ($f_1$ and $f_2$).
* **Sensor 4:** Highly un-correlated with all other channels, but exhibits an exceptionally large variance magnitude ($\sigma^2 \approx 2.0$) due to its high noise contamination scale.


### Question 1: The Total Variance Illusion (PCA Subplots 1 & 2 vs. FA Subplots 1 & 2)

* **Context:** In your PCA dashboard, `PC 1` captures roughly $45\%$ of the total system variance, and `PC 2` captures over $30\%$, indicating a highly structured, multi-component system. However, looking at the Factor Analysis dashboard, the **Variance Partitioning Stacked Bar** reveals that `Sensor_4` has a **Uniqueness value ($\varphi^2$) exceeding $98\%$**.
* **Core Problem:** 1. What does a Uniqueness value close to $100\%$ tell you about the physical nature of the data coming from `Sensor_4`?
2. Because PCA maximizes total global variance without distinguishing between types of variance, how does the massive localized noise of `Sensor_4` "trick" the PCA engine into inflating its top eigenvalues?
3. Why is this a major risk if an engineer relies *only* on PCA to build an automated plant anomaly detection framework?



### Question 2: Decoupling Structural Loading via Rotation (FA Subplot 1 vs. PCA Eigenvectors)

* **Context:** Look at the **Structural Loadings Matrix** heatmap in the top-left of the FA dashboard. You can see that `Sensor_1` and `Sensor_2` load heavily on `Factor 1`, while `Sensor_3` loads exclusively on `Factor 2`.
* **Core Problem:**
1. Traditional PCA forces its principal axes to follow a strict mathematical hierarchy ($\lambda_1 > \lambda_2 > \dots$) where every component tries to sweep up as much mixed variance as possible across all sensors. Explain how the **Varimax rotation** in Factor Analysis relaxes this rule to create "simple structure" (clean, isolated groupings).
2. From a plant operator's standpoint, why is the rotated FA heatmap much easier to troubleshoot during a structural failure than a raw, unrotated PCA eigenvector matrix?





### Question 3: Determining Subspace Truncation ($k$) using $T^2$ and $Q$ Profiles

* **Context:** Look at the right side of the PCA dashboard, showing **Mean Hotelling's $T^2$** and **Mean $Q$ Statistic (SPE)** plotted against the truncation cutoff $k$.
* **Core Problem:**
1. Describe the exact behavior of the Mean $Q$ Statistic curve as the subspace cutoff transitions from $k=1$ to $k=2$, and then from $k=2$ to $k=3$.
2. Why does a sharp drop followed by a distinct "flat elbow" in the $Q$ statistic identify **$k=2$** as the true hidden physical dimensionality of this asset?
3. If you incorrectly choose a truncation cutoff of $k=3$ for your monitoring system, what type of data are you forcing into your "clean" principal subspace?

### Question 4: Operational Trade-offs in System Health Monitoring

* **Context:** Imagine you are deploying a real-time predictive maintenance pipeline for a fleet of identical industrial assets based on these exact models.
* **Core Problem:** Compare the operational trade-offs of these two approaches:
1. **The PCA Strategy:** Tracking the main principal subspace using Hotelling's $T^2$ alongside a residual noise floor monitor using the $Q$ statistic.
2. **The FA Strategy:** Monitoring the rotated Latent Factor Scores directly.


* Which strategy is more robust against a single sensor losing calibration or experiencing an electrical short? Justify your choice using the **Sensor Uniqueness Noise Floor ($\varphi^2$)** metrics.